# Coin-selection project
## Python

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from collections import defaultdict

In [2]:
# Assuming files are stored in "saved-fees-updated" directory
data_dir = '../coin-selection-c-master/simulation/results'
files = os.listdir(data_dir)

# Predefined colors for each strategy
strategy_colors = defaultdict(lambda: 'black')
strategy_colors.update({
    'MAX_BILLS': 'blue',
    'CLOSEST_TO_EXPIRE_MIN_BILLS': 'orange',
    'CLOSEST_TO_EXPIRE_MAX_BILLS': 'cyan',
    'MAX_BILLS_TIME_TO_EXPIRE_WEIGHTED': 'yellow',
    'RANDOM': 'magenta',
    'MIN_BILLS': 'green',
    'EVEN_FROM_MIN_TO_MAX': 'red',
    'EVEN_FROM_MAX_TO_MIN': 'pink',
    'GREEDY_MIN_TO_MAX': 'purple',
    'GREEDY_MIN_TO_MAX_FIX': 'red'
})

# Initialize an empty list to hold all data
all_data = []

# Process each file
for file_name in files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'r') as file:
        # Read user type and strategy from the first line
        user_type, strategy = file.readline().strip().split(', ')
        
        # Parse the rest of the data
        for line in file:
            id, time, amount, operation, fee, coin_count = line.strip().split(', ')  # Updated here to include 'amount'
            all_data.append({
                "UserType": user_type,
                "Strategy": strategy,
                "Id": id,
                "Time": int(time),
                "Amount": int(amount),  
                "Operation": operation,  
                "Fee": int(fee),
                "FileName": file_name,
                "CoinCount": int(coin_count)
            })

# Convert the list of dictionaries to a DataFrame
raw_df = pd.DataFrame(all_data)

# Exclude DEPOSIT_REFRESH_OP so it wont be accounted for twice
df = raw_df.copy().loc[(raw_df['Operation'] != 'DEPOSIT_REFRESH_OP') & (raw_df['Amount'] > 0)]
df.describe()

ValueError: not enough values to unpack (expected 2, got 1)

# Plot cummulative fees, for all users/ strategies

In [ ]:
# Calculate cumulative fees for each file within each strategy
df['CumulativeFee'] = df.groupby(['FileName', 'UserType', 'Strategy'])['Fee'].cumsum()

# Create a pivot table for the final cumulative fee for each strategy and user type
final_cumulative_fees = df.drop_duplicates(['FileName', 'UserType', 'Strategy'], keep='last')
# Pivot table adjustment for easier access in the new plot setup
pivot_df_new = final_cumulative_fees.pivot_table(index='UserType', columns='Strategy', values='CumulativeFee', aggfunc=list)

# Plotting setup
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(50, 15))
axes = axes.ravel()

for idx, (user_type, ax) in enumerate(zip(pivot_df_new.index, axes)):
    # Prepare boxplot data; collect lists from each cell in row corresponding to user type
    boxplot_data = [pivot_df_new.loc[user_type, strategy] for strategy in pivot_df_new.columns]

    print(boxplot_data)
    
    ax.boxplot(boxplot_data, tick_labels=pivot_df_new.columns, showfliers=False)  # Optionally add `showfliers=False` to hide outliers
    ax.set_title(f"User Type: {user_type}", fontsize=26)
    ax.set_xlabel('Strategy', fontsize=22)
    ax.set_ylabel('Cumulative Fee', fontsize=22)
    ax.tick_params(axis='x', rotation=45, labelsize=13)

plt.tight_layout()
plt.savefig("results/final_cumulative_fees", bbox_inches='tight')
plt.show()

# Plot composition of Fees for each Strategy/User

In [ ]:

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Plotting setup
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(30, 30))
axes = axes.ravel()

for axis, strategy in zip(axes, strategy_colors):
    # Filter data for the current strategy
    strategy_data = df[df['Strategy'] == strategy].copy()

    # Calculate cumulative fees for each file within each strategy
    strategy_data['CumulativeFeeByOperation'] = strategy_data.groupby(['Operation', 'FileName', 'UserType', 'Strategy'])['Fee'].cumsum()

    final_cumulative_fees = strategy_data.drop_duplicates(['FileName', 'UserType', 'Strategy', 'Operation'], keep='last')
    
    barplot_data = final_cumulative_fees.sort_values(['Operation'])[['Operation', 'CumulativeFeeByOperation']]
    
    barplot_data.plot(ax=axis, kind='bar', x='Operation', y='CumulativeFeeByOperation', title=strategy,
                     xlabel="", ylabel="")
